### 01 — pybaseball smoke test

Purpose: Confirm `pybaseball` can pull Statcast data end-to-end against this 

Caching is enabled and pointed at `../data/cache/` so re-running this notebook is instant after the first pull.

In [1]:
from pathlib import Path

import pandas as pd
import pybaseball
from pybaseball import cache, statcast

DATA_DIR = Path("..") / "data" / "cache"
DATA_DIR.mkdir(parents=True, exist_ok=True)
cache.config.cache_directory = str(DATA_DIR.resolve())
cache.config.save()
cache.enable()

print("pybaseball:", pybaseball.__version__)
print("pandas    :", pd.__version__)
print("cache dir :", cache.config.cache_directory)

pybaseball: 2.2.7
pandas    : 2.2.3
cache dir : C:\Users\ihuang\Documents\python projects\MLB-Pitch-Predictor\data\cache


In [3]:
df = statcast("2024-06-15", "2024-06-15")
df.shape

This is a large query, it may take a moment to complete


  0%|          | 0/1 [00:00<?, ?it/s]c:\Users\ihuang\Documents\python projects\MLB-Pitch-Predictor\.venv\Lib\site-packages\pybaseball\datahelpers\postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
100%|██████████| 1/1 [00:07<00:00,  7.84s/it]


(4145, 118)

In [4]:
print("rows, cols:", df.shape)
print("\ndtype counts:")
print(df.dtypes.value_counts())
df.head(3)

rows, cols: (4145, 118)

dtype counts:
Int64             59
Float64           42
object            16
datetime64[ns]     1
Name: count, dtype: int64


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,...,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches
2195,SL,2024-06-15,90.0,2.77,6.13,"Banks, Tanner",682998,621383,field_out,hit_into_play,...,1,2.47,-0.2,-0.2,45.8,3.728685,5.508876,22.93471,38.355313,30.63759
2249,ST,2024-06-15,82.7,3.11,5.98,"Banks, Tanner",682998,621383,NaN,ball,...,1,3.76,-1.36,-1.36,42.5,<NA>,<NA>,<NA>,<NA>,<NA>
2336,SI,2024-06-15,93.6,2.72,6.24,"Banks, Tanner",682998,621383,NaN,blocked_ball,...,1,1.76,1.17,1.17,49.1,<NA>,<NA>,<NA>,<NA>,<NA>


In [5]:
print(f"{len(df.columns)} columns:")
for c in df.columns:
    print(f"  {c}")

118 columns:
  pitch_type
  game_date
  release_speed
  release_pos_x
  release_pos_z
  player_name
  batter
  pitcher
  events
  description
  spin_dir
  spin_rate_deprecated
  break_angle_deprecated
  break_length_deprecated
  zone
  des
  game_type
  stand
  p_throws
  home_team
  away_team
  type
  hit_location
  bb_type
  balls
  strikes
  game_year
  pfx_x
  pfx_z
  plate_x
  plate_z
  on_3b
  on_2b
  on_1b
  outs_when_up
  inning
  inning_topbot
  hc_x
  hc_y
  tfs_deprecated
  tfs_zulu_deprecated
  umpire
  sv_id
  vx0
  vy0
  vz0
  ax
  ay
  az
  sz_top
  sz_bot
  hit_distance_sc
  launch_speed
  launch_angle
  effective_speed
  release_spin_rate
  release_extension
  game_pk
  fielder_2
  fielder_3
  fielder_4
  fielder_5
  fielder_6
  fielder_7
  fielder_8
  fielder_9
  release_pos_y
  estimated_ba_using_speedangle
  estimated_woba_using_speedangle
  woba_value
  woba_denom
  babip_value
  iso_value
  launch_speed_angle
  at_bat_number
  pitch_number
  pitch_name
  home_scor

In [6]:
miss = df.isna().mean().sort_values(ascending=False)
miss[miss > 0].to_frame("pct_missing")

,pct_missing
spin_dir,1.000000
tfs_deprecated,1.000000
break_length_deprecated,1.000000
break_angle_deprecated,1.000000
sv_id,1.000000
...,...
az,0.002895
ay,0.002895
ax,0.002895
batter_days_until_next_game,0.002654


In [7]:
print("pitch_type missingness:", df["pitch_type"].isna().mean())
print("pitch_name missingness:", df["pitch_name"].isna().mean())
print("\npitch_type value counts:")
print(df["pitch_type"].value_counts(dropna=False))
print("\npitch_name value counts:")
print(df["pitch_name"].value_counts(dropna=False))

pitch_type missingness: 0.0028950542822677927
pitch_name missingness: 0.0028950542822677927

pitch_type value counts:
pitch_type
FF     1426
SI      609
SL      512
CH      422
CU      331
FC      318
ST      292
FS      166
KC       41
SV       16
NaN      12
Name: count, dtype: int64

pitch_name value counts:
pitch_name
4-Seam Fastball    1426
Sinker              609
Slider              512
Changeup            422
Curveball           331
Cutter              318
Sweeper             292
Split-Finger        166
Knuckle Curve        41
Slurve               16
NaN                  12
Name: count, dtype: int64
